In [1]:
from scipy.signal import find_peaks

In [7]:
from os import listdir
gpx_dir = 'C:\\Users\\banjanss\\OneDrive - UGent\\Documenten\\CyclingClusteringRanking\\NieuweDataCommunicatie\\'
files = listdir(gpx_dir)

In [8]:
len(files)

10360

In [4]:
gpx_dir+files[0]

'C:\\Users\\banjanss\\OneDrive - UGent\\Documenten\\CyclingClusteringRanking\\Scientific Data\\gpx_files\\gpx_before_dem\\2017 4 Jours de Dunkerque .gpx'

In [5]:
!pip install gpxpy

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ---------------------------------------- 0.0/42.6 kB ? eta -:--:--
   ---------------------------- ----------- 30.7/42.6 kB 660.6 kB/s eta 0:00:01
   ---------------------------------------- 42.6/42.6 kB 514.4 kB/s eta 0:00:00


In [35]:
gpx.tracks[0].segments[0].points

[GPXTrackPoint(47.34092, 8.71496, elevation=449.0, time=datetime.datetime(2025, 4, 4, 17, 34, 46, tzinfo=SimpleTZ('02:00'))),
 GPXTrackPoint(47.34008, 8.71461, elevation=448.0, time=datetime.datetime(2025, 4, 4, 17, 34, 55, tzinfo=SimpleTZ('02:00'))),
 GPXTrackPoint(47.33982, 8.7145, elevation=448.0, time=datetime.datetime(2025, 4, 4, 17, 34, 57, tzinfo=SimpleTZ('02:00'))),
 GPXTrackPoint(47.33972, 8.71446, elevation=448.0, time=datetime.datetime(2025, 4, 4, 17, 34, 57, tzinfo=SimpleTZ('02:00'))),
 GPXTrackPoint(47.33899, 8.71423, elevation=447.0, time=datetime.datetime(2025, 4, 4, 17, 35, 4, tzinfo=SimpleTZ('02:00'))),
 GPXTrackPoint(47.33896, 8.71422, elevation=447.0, time=datetime.datetime(2025, 4, 4, 17, 35, 4, tzinfo=SimpleTZ('02:00'))),
 GPXTrackPoint(47.33816, 8.71391, elevation=447.0, time=datetime.datetime(2025, 4, 4, 17, 35, 12, tzinfo=SimpleTZ('02:00'))),
 GPXTrackPoint(47.33762, 8.71365, elevation=446.0, time=datetime.datetime(2025, 4, 4, 17, 35, 17, tzinfo=SimpleTZ('02:00'

In [9]:
import gpxpy
from gpxpy import gpx
import pandas as pd
import numpy as np
from tqdm import tqdm

def get_number_peaks(series, prominence):
    store = find_peaks(series, prominence = prominence)
    try:
        if (len(store[0])>1)&(np.min(np.diff(store[0]))<prominence):
            return(len(store[1]['prominences'][np.diff(np.insert(store[0],0,0))>prominence]))
        else:
            return(len(store[1]['prominences']))
    except:
        return(len(store[1]['prominences']))

results = []
for file in tqdm(files):
    with open(gpx_dir+file, 'r', encoding="utf-8") as gpx_file:
        gpx = gpxpy.parse(gpx_file)
        track_elevation_series = []
        for data_point in gpx.tracks[0].segments[0].points:
            track_elevation_series.append(data_point.elevation)
        track_elevation_series.append(0)
        
        number_cat4 = get_number_peaks(track_elevation_series, 80)
        number_cat3 = get_number_peaks(track_elevation_series, 160)
        number_cat2 = get_number_peaks(track_elevation_series, 320)
        number_cat1 = get_number_peaks(track_elevation_series, 640)
        number_hc = get_number_peaks(track_elevation_series, 800)

        try:
            loc_last_cat4 = np.max(find_peaks(track_elevation_series, prominence = 80)[0])/len(track_elevation_series)
        except:
            loc_last_cat4 = 0
        try:
            loc_last_cat3 = np.max(find_peaks(track_elevation_series, prominence = 160)[0])/len(track_elevation_series)
        except:
            loc_last_cat3 = 0
        try:
            loc_last_cat2 = np.max(find_peaks(track_elevation_series, prominence = 320)[0])/len(track_elevation_series)
        except:
            loc_last_cat2 = 0
        try:
            loc_last_cat1 = np.max(find_peaks(track_elevation_series, prominence = 640)[0])/len(track_elevation_series)
        except:
            loc_last_cat1 = 0    
        try:
            loc_last_hc = np.max(find_peaks(track_elevation_series, prominence = 800)[0])/len(track_elevation_series)
        except:
            loc_last_hc = 0

        results.append([number_cat4, number_cat3, number_cat2, number_cat1, number_hc, loc_last_cat4, loc_last_cat3, loc_last_cat2, loc_last_cat1, loc_last_hc])    

100%|██████████| 10360/10360 [1:38:50<00:00,  1.75it/s] 


In [36]:
results2 = []
for file in tqdm(files):
    with open(gpx_dir+file, 'r', encoding="utf-8") as gpx_file:
        gpx = gpxpy.parse(gpx_file)
        gpx_track_points = gpx.tracks[0].segments[0].points
        elevations = [point.elevation for point in gpx_track_points]
        max_elevation = max(elevations)
        min_elevation = min(elevations)
        results2.append([max_elevation, min_elevation])

100%|██████████| 10360/10360 [1:33:59<00:00,  1.84it/s] 


In [ ]:
df2 = pd.DataFrame(results)
df2.columns = ['Number Category 4', 'Number Category 3', 'Number Category 2', 'Number Category 1', 'Number Hors Category', 'Location Last Category 4', 'Location Last Category 3', 'Location Last Category 2', 'Location Last Category 1', 'Location Last Hors Category',]
df2['Race Name'] = files

In [37]:
df_min_max = pd.DataFrame(results2)
df2[['Highest Elevation', 'Lowest Elevation']] = df_min_max


In [38]:
df2

,Number Category 4,Number Category 3,Number Category 2,Number Category 1,Number Hors Category,Location Last Category 4,Location Last Category 3,Location Last Category 2,Location Last Category 1,Location Last Hors Category,Race Name,Highest Elevation,Lowest Elevation
0,4,0,0,0,0,0.876576,0.000000,0.000000,0.0,0.0,1-etappe-obersterreichische-versicherung-junio...,430.0,300.0
1,3,2,1,0,0,0.999703,0.897070,0.388071,0.0,0.0,17-verona-valdobbiadene (1),399.0,13.0
2,3,2,1,0,0,0.999703,0.897070,0.388071,0.0,0.0,17-verona-valdobbiadene (2),399.0,13.0
3,3,2,1,0,0,0.999703,0.897070,0.388071,0.0,0.0,17-verona-valdobbiadene (3),399.0,13.0
4,3,2,1,0,0,0.999703,0.897070,0.388071,0.0,0.0,17-verona-valdobbiadene (4),399.0,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10355,5,5,0,0,0,0.918657,0.918657,0.000000,0.0,0.0,zurich-2024-mu-rr,668.0,405.0
10356,1,1,0,0,0,0.295812,0.295812,0.000000,0.0,0.0,zurich-2024-we-itt,641.0,406.0
10357,5,5,0,0,0,0.913889,0.913889,0.000000,0.0,0.0,zurich-2024-we-rr,669.0,406.0
10358,0,0,0,0,0,0.000000,0.000000,0.000000,0.0,0.0,zurich-2024-wj-itt,417.0,406.0


In [39]:
df2['Race Name'] = df2['Race Name'].str.replace('.gpx', '')

In [41]:
df2

,Number Category 4,Number Category 3,Number Category 2,Number Category 1,Number Hors Category,Location Last Category 4,Location Last Category 3,Location Last Category 2,Location Last Category 1,Location Last Hors Category,Race Name,Highest Elevation,Lowest Elevation
0,4,0,0,0,0,0.876576,0.000000,0.000000,0.0,0.0,1-etappe-obersterreichische-versicherung-junio...,430.0,300.0
1,3,2,1,0,0,0.999703,0.897070,0.388071,0.0,0.0,17-verona-valdobbiadene (1),399.0,13.0
2,3,2,1,0,0,0.999703,0.897070,0.388071,0.0,0.0,17-verona-valdobbiadene (2),399.0,13.0
3,3,2,1,0,0,0.999703,0.897070,0.388071,0.0,0.0,17-verona-valdobbiadene (3),399.0,13.0
4,3,2,1,0,0,0.999703,0.897070,0.388071,0.0,0.0,17-verona-valdobbiadene (4),399.0,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10355,5,5,0,0,0,0.918657,0.918657,0.000000,0.0,0.0,zurich-2024-mu-rr,668.0,405.0
10356,1,1,0,0,0,0.295812,0.295812,0.000000,0.0,0.0,zurich-2024-we-itt,641.0,406.0
10357,5,5,0,0,0,0.913889,0.913889,0.000000,0.0,0.0,zurich-2024-we-rr,669.0,406.0
10358,0,0,0,0,0,0.000000,0.000000,0.000000,0.0,0.0,zurich-2024-wj-itt,417.0,406.0


In [46]:
old = pd.read_csv('C:\\Users\\banjanss\\OneDrive - UGent\\Documenten\\CyclingClusteringRanking\\Scientific Data\\structured_course_data.csv')

In [47]:
new = pd.read_csv('KomootDataSurface2024_2025.csv')

In [48]:
rename_dict = {
    'name': 'Race Name',
    'Compacted gravel': 'Compacted Gravel',
    'ascending': 'Vertical Gain',
    'descending': 'Downhill',
    'lenght': 'Distance'
}

new['Net Gain'] = new['ascending']-new['descending']
new = new.rename(columns=rename_dict)

new = new.drop(columns = ['Unnamed: 0', 'NA_count', 'Unnamed: 10'])
old = old.drop(columns = ['Unnamed: 0', 'Highest Elevation', 'Lowest Elevation'])

In [49]:
np.setdiff1d(old.columns, new.columns)

array([], dtype=object)

In [50]:
old

,Race Name,Distance,Street,Road,Paved,Asphalt,Path,Cycleway,Unpaved,State Road,Cobblestones,Unknown,Compacted Gravel,Off-grid (unknown),Singletrack,Access Road,Alpine,Net Gain,Vertical Gain,Downhill
0,2022 A Travers les Hauts de France,117.440,2.44,115.0,11.50,106.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,900.0,850.0
1,2022 AG Tour de la Semois Stage 1,110.560,1.56,109.0,14.40,96.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-20.0,1660.0,1680.0
2,2022 AG Tour de la Semois Stage 2,113.260,5.55,101.0,11.60,101.0,2.68,4.030,1.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,1760.0,1710.0
3,2022 Ain Bugey Valromey Tour Stage 1,104.283,NaN,88.5,3.45,101.0,NaN,0.983,NaN,14.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,1110.0,1070.0
4,2022 Ain Bugey Valromey Tour Stage 2,98.060,NaN,96.1,NaN,98.0,NaN,NaN,NaN,1.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,950.0,930.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8089,2022 ZLM Tour Stage 1,101.360,8.68,83.6,5.69,84.0,NaN,9.080,NaN,NaN,11.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,120.0,120.0
8090,2022 ZLM Tour Stage 2,183.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,210.0,210.0
8091,2022 ZLM Tour Stage 3,192.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1350.0,1350.0
8092,2022 ZLM Tour Stage 4,198.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,360.0,340.0


In [51]:
new

,Race Name,Distance,Vertical Gain,Downhill,Cycleway,Street,Road,Paved,Asphalt,Cobblestones,State Road,Unpaved,Access Road,Unknown,Singletrack,Off-grid (unknown),Path,Compacted Gravel,Alpine,Net Gain
0,1-etappe-obersterreichische-versicherung-junio...,104.0,0.86000,0.86000,1.31,9.020,94.2,5.83,98.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00000
1,2-etappe-obersterreichische-versicherung-junio...,117.0,0.00187,0.00187,1.77,NaN,115.0,1.77,114.00,0.726,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00000
2,2024-acc-men-elite-rr,187.0,0.00293,0.00293,NaN,25.100,141.0,179.00,8.02,NaN,21.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00000
3,2024-acc-men-under-23-rr,140.0,0.00220,0.00220,NaN,15.600,106.0,134.00,6.01,NaN,18.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00000
4,2024-acc-women-elite-rr,105.0,0.00165,0.00165,NaN,14.100,79.1,101.00,4.51,NaN,11.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1845,giro-d-italia-2024-stage-17,159.0,0.00418,0.00416,1.33,0.577,70.5,26.50,131.00,0.592,86.00,0.25,NaN,1.01,0.171,0.167,0.751,NaN,NaN,0.00002
1846,giro-d-italia-2024-stage-18,179.0,0.97000,0.00163,11.20,4.750,101.0,17.50,160.00,0.511,60.90,0.00,NaN,1.54,0.000,0.774,0.776,NaN,NaN,0.96837
1847,giro-d-italia-2024-stage-20,184.0,0.00429,0.00456,3.93,1.250,154.0,13.20,168.00,NaN,22.10,NaN,NaN,2.85,NaN,2.850,NaN,NaN,NaN,-0.00027
1848,giro-d-italia-2024-stage-21,129.0,0.93000,0.92000,1.70,8.650,71.6,1.90,96.40,21.600,21.30,NaN,NaN,9.09,10.100,8.550,7.000,NaN,NaN,0.01000


In [52]:
full = pd.concat([old, new])

In [64]:
cluster_df = full[-full['Race Name'].str.contains(r"\s\(\d+\)$")]

In [60]:


cluster_df

,Number Category 4,Number Category 3,Number Category 2,Number Category 1,Number Hors Category,Location Last Category 4,Location Last Category 3,Location Last Category 2,Location Last Category 1,Location Last Hors Category,...,Cobblestones,Unknown,Compacted Gravel,Off-grid (unknown),Singletrack,Access Road,Alpine,Net Gain,Vertical Gain,Downhill
0,4,0,0,0,0,0.876576,0.000000,0.000000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0.86000,0.86000
1,3,2,1,0,0,0.999703,0.897070,0.388071,0.0,0.0,...,0.136,5.230,NaN,5.130,0.127,0.0,NaN,-0.83897,0.00103,0.84000
2,5,5,0,0,0,0.966202,0.966202,0.000000,0.0,0.0,...,0.726,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0.00187,0.00187
3,1,0,0,0,0,0.500632,0.000000,0.000000,0.0,0.0,...,5.230,0.831,NaN,0.831,NaN,NaN,NaN,40.00000,690.00000,650.00000
4,8,8,0,0,0,0.791565,0.791565,0.000000,0.0,0.0,...,0.122,0.366,NaN,0.366,NaN,NaN,NaN,-10.00000,1880.00000,1890.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9936,5,5,0,0,0,0.918657,0.918657,0.000000,0.0,0.0,...,0.654,0.000,0.102,NaN,NaN,NaN,NaN,-0.00004,0.00228,0.00232
9937,1,1,0,0,0,0.295812,0.295812,0.000000,0.0,0.0,...,NaN,1.400,NaN,NaN,NaN,NaN,NaN,-0.06000,0.32000,0.38000
9938,5,5,0,0,0,0.913889,0.913889,0.000000,0.0,0.0,...,0.751,0.000,0.000,NaN,NaN,NaN,NaN,-0.00005,0.00218,0.00223
9939,0,0,0,0,0,0.000000,0.000000,0.000000,0.0,0.0,...,NaN,0.000,NaN,NaN,NaN,NaN,NaN,-0.01000,0.05000,0.06000


In [65]:
cluster_df['Race Name'][cluster_df['Race Name'].duplicated(keep=False)].unique()

array([], dtype=object)

In [66]:
cluster_df.shape

(9944, 20)

In [67]:
df2.shape

(10360, 13)

In [68]:
cluster_df = df2.merge(cluster_df, on = 'Race Name')

In [69]:
cluster_df.shape

(9941, 32)

In [70]:
cluster_df['Lowest Elevation'][cluster_df['Lowest Elevation'].isna()] = 0
cluster_df['Highest Elevation'][cluster_df['Highest Elevation'].isna()] = 0
cluster_df['Vertical Gain'][cluster_df['Vertical Gain'].isna()] = 0
cluster_df['Downhill'][cluster_df['Downhill'].isna()] = 0

C:\Users\banjanss\AppData\Local\Temp\ipykernel_40524\782869186.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  cluster_df['Lowest Elevation'][cluster_df['Lowest Elevation'].isna()] = 0
C:\Users\banjanss\AppData\Local\Temp\ipykernel_40524

In [71]:
cluster_df['nas'] = cluster_df.isnull().sum(axis=1)

In [74]:
structured_missing = cluster_df[cluster_df['nas']<15]
all_missing = cluster_df[cluster_df['nas']>=15]

In [75]:
structured_missing = structured_missing.fillna(0)

In [76]:
one_day = all_missing[-all_missing['Race Name'].str.contains('Stage')]
stages = all_missing[all_missing['Race Name'].str.contains('Stage')]

In [77]:
one_day = one_day.reset_index(drop = True)

In [78]:
for index, row in one_day.iterrows():
    race = ''.join([i for i in row['Race Name'] if not i.isdigit()])
    if 'Vlaanderen' in race:
        print('Werkt!')
        race='Vlaanderen'
    other_editions = structured_missing[structured_missing['Race Name'].str.contains(race)]
    if other_editions.shape[0]>0:
        col_to_impute = other_editions.columns[row.isna()]
        one_day.loc[index, col_to_impute] = other_editions[col_to_impute].mean()

Werkt!
Werkt!
Werkt!


C:\Users\banjanss\AppData\Local\Temp\ipykernel_40524\3136770471.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  other_editions = structured_missing[structured_missing['Race Name'].str.contains(race)]


Werkt!
Werkt!
Werkt!
Werkt!


C:\Users\banjanss\AppData\Local\Temp\ipykernel_40524\3136770471.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  other_editions = structured_missing[structured_missing['Race Name'].str.contains(race)]


Werkt!
Werkt!
Werkt!
Werkt!
Werkt!
Werkt!
Werkt!


C:\Users\banjanss\AppData\Local\Temp\ipykernel_40524\3136770471.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  other_editions = structured_missing[structured_missing['Race Name'].str.contains(race)]


In [79]:
one_day.isna().sum()

Number Category 4                0
Number Category 3                0
Number Category 2                0
Number Category 1                0
Number Hors Category             0
Location Last Category 4         0
Location Last Category 3         0
Location Last Category 2         0
Location Last Category 1         0
Location Last Hors Category      0
Race Name                        0
Highest Elevation                0
Lowest Elevation                 0
Distance                         4
Street                         204
Road                           204
Paved                          204
Asphalt                        204
Path                           204
Cycleway                       204
Unpaved                        204
State Road                     204
Cobblestones                   204
Unknown                        204
Compacted Gravel               204
Off-grid (unknown)             204
Singletrack                    204
Access Road                    204
Alpine              

In [80]:
df = pd.concat([structured_missing, one_day, stages])

In [81]:
df = df.reset_index()

In [82]:
df.columns

Index(['index', 'Number Category 4', 'Number Category 3', 'Number Category 2',
       'Number Category 1', 'Number Hors Category', 'Location Last Category 4',
       'Location Last Category 3', 'Location Last Category 2',
       'Location Last Category 1', 'Location Last Hors Category', 'Race Name',
       'Highest Elevation', 'Lowest Elevation', 'Distance', 'Street', 'Road',
       'Paved', 'Asphalt', 'Path', 'Cycleway', 'Unpaved', 'State Road',
       'Cobblestones', 'Unknown', 'Compacted Gravel', 'Off-grid (unknown)',
       'Singletrack', 'Access Road', 'Alpine', 'Net Gain', 'Vertical Gain',
       'Downhill', 'nas'],
      dtype='object')

In [84]:
race_names = df['Race Name']
df = df.drop(columns = ['Race Name', 'nas'])

In [85]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=3)
imputed_df = imputer.fit_transform(df)

In [86]:
race_names

0       1-etappe-obersterreichische-versicherung-junio...
1                                 17-verona-valdobbiadene
2       2-etappe-obersterreichische-versicherung-junio...
3                              2017 4 Jours de Dunkerque 
4                   2017 99a Coppa Bernocchi - 42o GP BPM
                              ...                        
9936               2022 XXVIII. Gipuzkoa Klasikoa Stage 2
9937                                2022 ZLM Tour Stage 2
9938                                2022 ZLM Tour Stage 3
9939                                2022 ZLM Tour Stage 4
9940                                2022 ZLM Tour Stage 5
Name: Race Name, Length: 9941, dtype: object

In [87]:
imputed_df

array([[ 0.00000000e+00,  4.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  8.60000000e-01,  8.60000000e-01],
       [ 1.00000000e+00,  3.00000000e+00,  2.00000000e+00, ...,
        -8.38970000e-01,  1.03000000e-03,  8.40000000e-01],
       [ 2.00000000e+00,  5.00000000e+00,  5.00000000e+00, ...,
         0.00000000e+00,  1.87000000e-03,  1.87000000e-03],
       ...,
       [ 6.72300000e+03,  4.00000000e+00,  1.00000000e+00, ...,
         5.66666667e+01,  1.35000000e+03,  1.35000000e+03],
       [ 6.72400000e+03,  0.00000000e+00,  0.00000000e+00, ...,
         2.00000000e+01,  3.60000000e+02,  3.40000000e+02],
       [ 6.72500000e+03,  0.00000000e+00,  0.00000000e+00, ...,
         1.00000000e+01,  2.60000000e+02,  2.50000000e+02]])

In [89]:
label_list = []

import time
import warnings

import matplotlib.pyplot as plt
from sklearn import cluster, datasets, mixture
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler
from itertools import cycle, islice

np.random.seed(0)



# ============
# Set up cluster parameters
# ============
plt.figure(figsize=(9 * 2 + 3, 13))
plt.subplots_adjust(
    left=0.02, right=0.98, bottom=0.001, top=0.95, wspace=0.05, hspace=0.01
)

plot_num = 1

default_base = {
    "quantile": 0.3,
    "eps": 0.18,
    "damping": 0.9,
    "preference": -200,
    "n_neighbors": 2,
    "min_samples": 30,
    "xi": 0.05,
    "min_cluster_size": 30,
}


num_clust = 9 
    # update parameters with dataset-specific values
params = default_base.copy()

X = pd.DataFrame(imputed_df)

# normalize dataset for easier parameter selection
X = StandardScaler().fit_transform(X)

# estimate bandwidth for mean shift
bandwidth = cluster.estimate_bandwidth(X, quantile=params["quantile"])

# connectivity matrix for structured Ward
connectivity = kneighbors_graph(
    X, n_neighbors=params["n_neighbors"], include_self=False
)
# make connectivity symmetric
connectivity = 0.5 * (connectivity + connectivity.T)

two_means = cluster.MiniBatchKMeans(n_clusters=num_clust, batch_size = 2048)

algorithm = two_means

t0 = time.time()

# catch warnings related to kneighbors_graph
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="the number of connected components of the "
        + "connectivity matrix is [0-9]{1,2}"
        + " > 1. Completing it to avoid stopping the tree early.",
        category=UserWarning,
    )
    warnings.filterwarnings(
        "ignore",
        message="Graph is not fully connected, spectral embedding"
        + " may not work as expected.",
        category=UserWarning,
    )
    algorithm.fit(X)

t1 = time.time()
if hasattr(algorithm, "labels_"):
    y_pred = algorithm.labels_.astype(int)
else:
    y_pred = algorithm.predict(X)

c:\Users\banjanss\Anaconda3\envs\py_lectures\Lib\site-packages\sklearn\cluster\_kmeans.py:1952: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 2560 or by setting the environment variable OMP_NUM_THREADS=8
  warnings.warn(


<Figure size 2100x1300 with 0 Axes>

In [117]:
clustered_races = pd.DataFrame({'Race Name': race_names, 'Cluster': y_pred})

In [ ]:
clustered_races.to_csv('ClustersBeforeRoubaix2025.csv', index = False)

: 

In [91]:
y_pred

array([8, 4, 3, ..., 6, 0, 0])

In [111]:
for name in race_names[y_pred==8]:
    print(name)

1-etappe-obersterreichische-versicherung-junioren-rundfahrt-2024
2017 ACC Asian Road Championships - ITT
2017 ACC Asian Road Championships - TTT
2017 ACC Asian Road Championships
2017 Amgen Tour of California Stage 6
2017 Australian Road National Championships - ITT
2017 Baloise Belgium Tour Stage 3
2017 Belgian Road National Championships - ITT
2017 BeNe Ladies Tour Prologue
2017 BeNe Ladies Tour Stage 2-2
2017 Boels Rentals Ladies Tour Prologue
2017 Boucles de la Mayenne Prologue
2017 British Road National Championships - ITT
2017 CAC African Road Championships - ITT
2017 CAC African Road Championships - TTT
2017 CAC African Road Championships
2017 Canadian Road National Championships - ITT
2017 Canadian Road National Championships
2017 Chrono des Nations
2017 Circuit Cycliste Sarthe - Pays de la Loire Stage 2-1
2017 Circuit Cycliste Sarthe - Pays de la Loire Stage 2-2
2017 COPACI Campeonato Panamericano de Ruta - ITT
2017 COPACI Campeonato Panamericano de Ruta
2017 Crescent Vargarda

In [108]:
y_pred[race_names.str.contains('Liege')]

array([1, 1, 3, 1, 1, 1, 3, 1, 3, 1, 3, 1, 3, 3, 3])

In [269]:
#annotation = pd.read_csv('named_annotation_NOV23.csv')

In [532]:
annotation = pd.read_excel('FinalAnnotationNOV2023.xlsx')

In [583]:
annotation = annotation[(annotation['name2_x'].isin(race_names))&(annotation['name2_y'].isin(race_names))]

In [581]:
race_names

0                   2017 4 Jours de Dunkerque 
1        2017 99a Coppa Bernocchi - 42o GP BPM
2                  2017 Abu Dhabi Tour Stage 1
3                  2017 Abu Dhabi Tour Stage 3
4                  2017 Abu Dhabi Tour Stage 4
                         ...                  
6227    2022 XXVIII. Gipuzkoa Klasikoa Stage 2
6228                     2022 ZLM Tour Stage 2
6229                     2022 ZLM Tour Stage 3
6230                     2022 ZLM Tour Stage 4
6231                     2022 ZLM Tour Stage 5
Name: Race Name, Length: 6232, dtype: object

In [587]:
annotation = annotation.reset_index()

In [588]:
import random

#Nieuwe split
test_index = random.sample(list(annotation.index), 100)
train_index = list(set(list(annotation.index)).difference(set(test_index)))

train_set = annotation.iloc[train_index]
test_set = annotation.iloc[test_index]

In [428]:
#train_set = annotation
#test_set = annotation

In [589]:
test_set['Label'].value_counts()

MustLink      56
CannotLink    44
Name: Label, dtype: int64

In [590]:
weight_cannot = test_set['Label'].value_counts()['MustLink']/test_set['Label'].value_counts()['CannotLink']

In [342]:
#annotation[['name2_x', 'name2_y', 'Label']].to_excel('FinalAnnotationNOV2023.xlsx', index = False)

In [591]:
merged_set_constrained2 = pd.concat([race_names,pd.DataFrame(np.matrix(label_list).transpose())], axis = 1)
evaluation_constrained2 = test_set[['name2_x', 'name2_y', 'Label']].merge(merged_set_constrained2, left_on = 'name2_x', right_on = 'Race Name').merge(merged_set_constrained2, left_on = 'name2_y', right_on = 'Race Name')

scores_constrained2 = []

for i in range(70):
    score = 0
    clust1 = str(i)+'_x'
    clust2 = str(i)+'_y'
    for j in range(evaluation_constrained2.shape[0]):
        subject = evaluation_constrained2.iloc[j,:]
        if subject[clust1]==subject[clust2]:
            if subject['Label']=='MustLink':
                score+=-1
            else:
                score+=10
        else:
            if subject['Label']=='CannotLink':
                score+=-2
            else:
                score+=5 
                
    scores_constrained2.append(score)

In [592]:
pd.DataFrame(np.array(scores_constrained2).reshape((7,10)).transpose())

,0,1,2,3,4,5,6
0,-54,-96,-60,12,-60,-114,-114
1,-36,-36,-36,-36,-36,-36,-36
2,354,354,354,354,354,354,354
3,54,-48,-48,-36,18,-6,-24
4,-60,-60,-102,-96,-96,-96,-96
5,384,384,384,384,384,384,384
6,384,384,384,384,384,384,384
7,384,384,384,384,384,384,384
8,-36,-36,-36,-30,-30,-30,-60
9,72,114,84,90,120,96,42


In [618]:
train_set

,index,name2_x,name2_y,Label
0,0,2017 Dwars Door Vlaanderen,2017 Ronde Van Vlaanderen,MustLink
3,3,2018 Tirreno-Adriatico Stage 5,2019 Tirreno-Adriatico Stage 4,MustLink
4,4,2018 Ronde van Vlaanderen Beloften,2018 Ronde van Vlaanderen Juniores,MustLink
7,7,2018 Ronde van Vlaanderen Juniores,2018 Ronde Van Vlaanderen,MustLink
9,9,2019 Tirreno-Adriatico Stage 5,2020 Tirreno Adriatico Stage 7,MustLink
...,...,...,...,...
187,194,2017 Vuelta a Espana Stage 12,2017 Vuelta a Espana Stage 1,CannotLink
190,197,2017 Vuelta al Pais Vasco Stage 2,2017 Vuelta Ciclista a la Provincia de San Jua...,CannotLink
194,201,2021 Vuelta a Colombia Stage 1,2022 Tour of Norway Stage 1,CannotLink
197,204,2017 Vuelta a Espana Stage 20,2022 Paris-Roubaix,CannotLink


In [593]:
scores_constrained_bad_metric = []

for i in range(70):
    score = 0
    clust1 = str(i)+'_x'
    clust2 = str(i)+'_y'
    for j in range(evaluation_constrained2.shape[0]):
        subject = evaluation_constrained2.iloc[j,:]
        if subject[clust1]==subject[clust2]:
            if subject['Label']=='MustLink':
                score+=1
        else:
            if subject['Label']=='CannotLink':
                score+=1
                
    scores_constrained_bad_metric.append(score)

In [594]:
pd.DataFrame(np.array(scores_constrained_bad_metric).reshape((7,10)).transpose())

,0,1,2,3,4,5,6
0,92,95,90,80,88,95,96
1,82,82,82,82,82,82,82
2,57,57,57,57,57,57,57
3,82,89,89,87,80,82,84
4,92,92,95,94,94,94,94
5,56,56,56,56,56,56,56
6,56,56,56,56,56,56,56
7,56,56,56,56,56,56,56
8,90,90,90,89,89,89,91
9,78,75,72,71,67,70,73


In [628]:
pd.DataFrame(np.array(scores_constrained_bad_metric).reshape((7,10)).transpose()).to_excel('ExternalClusValidationNOV23.xlsx')

In [458]:
np.min(scores_constrained2)

422

In [301]:
pd.DataFrame(np.array(scores_constrained2).reshape((7,10)).transpose()).to_excel('ExternalClusValidationNOV23.xlsx')

In [631]:
from collections import Counter
scores = []
for i in range(70):
    metric = []
    for key, value in Counter(label_list[i]).items():
        metric.append(value/len(label_list[i]))
    scores.append(np.max(metric))  

In [633]:
pd.DataFrame(np.array(scores).reshape((7,10)).transpose()).to_excel('PercentageLargestClusterNOV23.xlsx')

In [632]:
pd.DataFrame(np.array(scores).reshape((7,10)).transpose())

,0,1,2,3,4,5,6
0,0.550225,0.452503,0.318357,0.333761,0.272625,0.229461,0.180680
1,0.048299,0.048299,0.048299,0.048299,0.048299,0.048299,0.048299
2,0.979140,0.979140,0.979140,0.979140,0.979140,0.979140,0.979140
3,0.744384,0.510591,0.477214,0.492779,0.616175,0.320603,0.437420
4,0.483312,0.483312,0.361200,0.361200,0.361200,0.361200,0.361200
5,0.999679,0.998716,0.998074,0.997914,0.997272,0.996309,0.996149
6,0.999037,0.999037,0.999037,0.999037,0.999037,0.999037,0.999037
7,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
8,0.439185,0.439185,0.439185,0.422978,0.422978,0.422978,0.383184
9,0.593870,0.601573,0.511072,0.441431,0.386393,0.341142,0.314827


In [612]:
import random

def cop_kmeans(dataset, k, ml=[], cl=[],
               initialization='kmpp',
               max_iter=100, tol=0):

    ml, cl = transitive_closure(ml, cl, len(dataset))
    ml_info = get_ml_info(ml, dataset)
    tol = tolerance(tol, dataset)

    centers = initialize_centers(dataset, k, initialization)

    for _ in range(max_iter):
        clusters_ = [-1] * len(dataset)
        for i, d in enumerate(dataset):
            indices, _ = closest_clusters(centers, d)
            counter = 0
            if clusters_[i] == -1:
                found_cluster = False
                while (not found_cluster) and counter < len(indices):
                    index = indices[counter]
                    if not violate_constraints(i, index, clusters_, ml, cl):
                        found_cluster = True
                        clusters_[i] = index
                        for j in ml[i]:
                            clusters_[j] = index
                    counter += 1

                if not found_cluster:
                    return None, None

        clusters_, centers_ = compute_centers(clusters_, dataset, k, ml_info)
        shift = sum(l2_distance(centers[i], centers_[i]) for i in range(k))
        if shift <= tol:
            break

        centers = centers_

    return clusters_, centers_

def l2_distance(point1, point2):
    return np.sum([(float(i)-float(j))**2 for (i, j) in zip(point1, point2)])

# taken from scikit-learn (https://goo.gl/1RYPP5)
def tolerance(tol, dataset):
    n = len(dataset)
    dim = len(dataset[0])
    averages = [sum(dataset[i][d] for i in range(n))/float(n) for d in range(dim)]
    variances = [sum((dataset[i][d]-averages[d])**2 for i in range(n))/float(n) for d in range(dim)]
    return tol * sum(variances) / dim

def closest_clusters(centers, datapoint):
    distances = [l2_distance(center, datapoint) for
                 center in centers]
    return sorted(range(len(distances)), key=lambda x: distances[x]), distances

def initialize_centers(dataset, k, method):
    if method == 'random':
        ids = list(range(len(dataset)))
        random.shuffle(ids)
        return [dataset[i] for i in ids[:k]]

    elif method == 'kmpp':
        chances = [1] * len(dataset)
        centers = []

        for _ in range(k):
            chances = [x/sum(chances) for x in chances]
            r = random.random()
            acc = 0.0
            for index, chance in enumerate(chances):
                if acc + chance >= r:
                    break
                acc += chance
            centers.append(dataset[index])

            for index, point in enumerate(dataset):
                cids, distances = closest_clusters(centers, point)
                chances[index] = distances[cids[0]]

        return centers

def violate_constraints(data_index, cluster_index, clusters, ml, cl):
    for i in ml[data_index]:
        if clusters[i] != -1 and clusters[i] != cluster_index:
            return True

    for i in cl[data_index]:
        if clusters[i] == cluster_index:
            return True

    return False

def compute_centers(clusters, dataset, k, ml_info):
    cluster_ids = set(clusters)
    k_new = len(cluster_ids)
    id_map = dict(zip(cluster_ids, range(k_new)))
    clusters = [id_map[x] for x in clusters]

    dim = len(dataset[0])
    centers = [[0.0] * dim for i in range(k)]

    counts = [0] * k_new
    for j, c in enumerate(clusters):
        for i in range(dim):
            centers[c][i] += dataset[j][i]
        counts[c] += 1

    for j in range(k_new):
        for i in range(dim):
            centers[j][i] = centers[j][i]/float(counts[j])

    if k_new < k:
        ml_groups, ml_scores, ml_centroids = ml_info
        current_scores = [sum(l2_distance(centers[clusters[i]], dataset[i])
                              for i in group)
                          for group in ml_groups]
        group_ids = sorted(range(len(ml_groups)),
                           key=lambda x: current_scores[x] - ml_scores[x],
                           reverse=True)

        for j in range(k-k_new):
            gid = group_ids[j]
            cid = k_new + j
            centers[cid] = ml_centroids[gid]
            for i in ml_groups[gid]:
                clusters[i] = cid

    return clusters, centers

def get_ml_info(ml, dataset):
    flags = [True] * len(dataset)
    groups = []
    for i in range(len(dataset)):
        if not flags[i]: continue
        group = list(ml[i] | {i})
        groups.append(group)
        for j in group:
            flags[j] = False

    dim = len(dataset[0])
    scores = [0.0] * len(groups)
    centroids = [[0.0] * dim for i in range(len(groups))]

    for j, group in enumerate(groups):
        for d in range(dim):
            for i in group:
                centroids[j][d] += dataset[i][d]
            centroids[j][d] /= float(len(group))

    scores = [sum(l2_distance(centroids[j], dataset[i])
                  for i in groups[j])
              for j in range(len(groups))]

    return groups, scores, centroids

def transitive_closure(ml, cl, n):
    ml_graph = dict()
    cl_graph = dict()
    for i in range(n):
        ml_graph[i] = set()
        cl_graph[i] = set()

    def add_both(d, i, j):
        d[i].add(j)
        d[j].add(i)

    for (i, j) in ml:
        add_both(ml_graph, i, j)

    def dfs(i, graph, visited, component):
        visited[i] = True
        for j in graph[i]:
            if not visited[j]:
                dfs(j, graph, visited, component)
        component.append(i)

    visited = [False] * n
    for i in range(n):
        if not visited[i]:
            component = []
            dfs(i, ml_graph, visited, component)
            for x1 in component:
                for x2 in component:
                    if x1 != x2:
                        ml_graph[x1].add(x2)
    for (i, j) in cl:
        add_both(cl_graph, i, j)
        for y in ml_graph[j]:
            add_both(cl_graph, i, y)
        for x in ml_graph[i]:
            add_both(cl_graph, x, j)
            for y in ml_graph[j]:
                add_both(cl_graph, x, y)

    for i in ml_graph:
        for j in ml_graph[i]:
            if j != i and j in cl_graph[i]:
                raise Exception('inconsistent constraints between %d and %d' %(i, j))

    return ml_graph, cl_graph

In [613]:
must_link = train_set[train_set['Label']=='MustLink']
cannot_link = train_set[train_set['Label']=='CannotLink']

nieuwe_tussen = race_names
nieuwe_tussen = pd.DataFrame(nieuwe_tussen)
nieuwe_tussen['index2'] = range(nieuwe_tussen.shape[0])

cannot_link_final = cannot_link[['name2_x', 'name2_y']].merge(nieuwe_tussen, left_on = 'name2_x', right_on = 'Race Name').merge(nieuwe_tussen, left_on = 'name2_y', right_on = 'Race Name')[['index2_x', 'index2_y']]
must_link_final = must_link[['name2_x', 'name2_y']].merge(nieuwe_tussen, left_on = 'name2_x', right_on = 'Race Name').merge(nieuwe_tussen, left_on = 'name2_y', right_on = 'Race Name')[['index2_x', 'index2_y']]

must_link_final = list(zip(must_link_final['index2_x'], must_link_final['index2_y']))
cannot_link_final = list(zip(cannot_link_final['index2_x'], cannot_link_final['index2_y']))

results_list_constrained = []
for k_set in range(3,11):
    clusters, centers = cop_kmeans(dataset=np.array(np.matrix(X)), k=k_set, ml=must_link_final,cl=cannot_link_final)
    results_list_constrained.append(clusters)

In [624]:
merged_set_constrained3 = pd.concat([race_names,pd.DataFrame(np.matrix(results_list_constrained).transpose())], axis = 1)
evaluation_constrained3 = test_set[['name2_x', 'name2_y', 'Label']].merge(merged_set_constrained3, left_on = 'name2_x', right_on = 'Race Name').merge(merged_set_constrained3, left_on = 'name2_y', right_on = 'Race Name')

scores_cop_kmeans = []

for i in range(6):
    score = 0
    clust1 = str(i)+'_x'
    clust2 = str(i)+'_y'
    for j in range(evaluation_constrained3.shape[0]):
        subject = evaluation_constrained3.iloc[j,:]
        if subject[clust1]==subject[clust2]:
            if subject['Label']=='MustLink':
                score+=-1
            else:
                score+=10
        else:
            if subject['Label']=='CannotLink':
                score+=-2
            else:
                score+=5 
                
    scores_cop_kmeans.append(score)

In [625]:
scores_cop_kmeans

[156, 156, -96, -90, -72, -78]

In [626]:
scores_cop_kmeans_other_metric = []

for i in range(7):
    score = 0
    clust1 = str(i)+'_x'
    clust2 = str(i)+'_y'
    for j in range(evaluation_constrained3.shape[0]):
        subject = evaluation_constrained3.iloc[j,:]
        if subject[clust1]==subject[clust2]:
            if subject['Label']=='MustLink':
                score+=1
        else:
            if subject['Label']=='CannotLink':
                score+=1
                
    scores_cop_kmeans_other_metric.append(score)

In [627]:
scores_cop_kmeans_other_metric

[69, 69, 95, 91, 93, 90, 90]

In [550]:
train_set

,name2_x,name2_y,Label
0,2017 Dwars Door Vlaanderen,2017 Ronde Van Vlaanderen,MustLink
2,2017 Ronde Van Vlaanderen,2019 Tirreno-Adriatico Stage 4,CannotLink
3,2018 Tirreno-Adriatico Stage 5,2019 Tirreno-Adriatico Stage 4,MustLink
4,2018 Ronde van Vlaanderen Beloften,2018 Ronde van Vlaanderen Juniores,MustLink
5,2018 Ronde van Vlaanderen Beloften,2019 Tirreno-Adriatico Stage 5,CannotLink
...,...,...,...
201,2021 Vuelta a Colombia Stage 1,2022 Tour of Norway Stage 1,CannotLink
203,2017 Vuelta a Espana Stage 20,2022 Paris-Roubaix,CannotLink
204,2017 Vuelta a Espana Stage 20,2022 Paris-Roubaix,CannotLink
205,2022 Tour of Slovenia Stage 2,2022 Paris-Roubaix,CannotLink


In [551]:
test_set

,name2_x,name2_y,Label
101,2020 Il Lombardia,2019 Paris-Nice Stage 5,CannotLink
200,2022 Tour of Slovenia Stage 2,2022 Tour of Norway Stage 1,CannotLink
111,2017 Clasica Ciclista San Sebastian,2018 Clasica Ciclista San Sebastian,MustLink
126,2020 Bretagne Classic - Ouest France,2021 Bretagne Classic - Ouest-France,MustLink
60,2018 Tre Valli Varesine,2019 Tre Valli Varesine,MustLink
...,...,...,...
9,2019 Tirreno-Adriatico Stage 5,2020 Tirreno Adriatico Stage 7,MustLink
151,2021 Tour de Hongrie Stage 5,2022 GP Internacional Torres Vedras - Trofeu J...,CannotLink
172,2017 Driedaagse De Panne - Koksijde Stage 3-1,2017 Driedaagse De Panne - Koksijde Stage 3-2,CannotLink
8,2018 Ronde van Vlaanderen Juniores,2020 Tirreno Adriatico Stage 7,CannotLink


In [548]:
evaluation_constrained3

,name2_x,name2_y,Label,Race Name_x,0_x,1_x,2_x,3_x,4_x,5_x,Race Name_y,0_y,1_y,2_y,3_y,4_y,5_y
0,2020 Il Lombardia,2019 Paris-Nice Stage 5,CannotLink,2020 Il Lombardia,0,2,3,5,1,5,2019 Paris-Nice Stage 5,2,0,0,3,0,1
1,2020 Il Lombardia,2021 Il Lombardia,MustLink,2020 Il Lombardia,0,2,3,5,1,5,2021 Il Lombardia,0,2,3,5,1,5
2,2022 Tour of Slovenia Stage 2,2022 Tour of Norway Stage 1,CannotLink,2022 Tour of Slovenia Stage 2,0,3,4,0,6,0,2022 Tour of Norway Stage 1,2,2,3,5,2,8
3,2017 Clasica Ciclista San Sebastian,2018 Clasica Ciclista San Sebastian,MustLink,2017 Clasica Ciclista San Sebastian,0,3,4,0,2,8,2018 Clasica Ciclista San Sebastian,0,2,3,5,1,5
4,2020 Bretagne Classic - Ouest France,2021 Bretagne Classic - Ouest-France,MustLink,2020 Bretagne Classic - Ouest France,2,3,4,0,2,0,2021 Bretagne Classic - Ouest-France,0,3,4,0,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,2020 UAE Tour Stage 3,2017 Coppa Agostoni - Giro delle Brianze,CannotLink,2020 UAE Tour Stage 3,2,2,3,5,1,4,2017 Coppa Agostoni - Giro delle Brianze,0,3,4,0,2,8
92,2017 Tirreno-Adriatico Stage 3,2017 Tirreno-Adriatico Stage 4,CannotLink,2017 Tirreno-Adriatico Stage 3,0,3,4,0,2,8,2017 Tirreno-Adriatico Stage 4,0,2,3,5,1,4
93,2018 Il Lombardia,2021 Giro d'Italia Stage 21,CannotLink,2018 Il Lombardia,0,2,3,5,1,5,2021 Giro d'Italia Stage 21,2,0,0,3,0,1
94,2017 Driedaagse De Panne - Koksijde Stage 3-1,2017 Driedaagse De Panne - Koksijde Stage 3-2,CannotLink,2017 Driedaagse De Panne - Koksijde Stage 3-1,2,0,0,3,0,7,2017 Driedaagse De Panne - Koksijde Stage 3-2,2,0,0,3,0,1


In [603]:
finale_clus = pd.DataFrame(np.matrix([race_names, results_list_constrained[5], label_list[60]]).transpose())

In [604]:
finale_clus.columns = ['Race Name', 'Constrained', 'KmeansClust']

In [611]:
finale_clus.to_csv('clustersNOV2023.csv', index = False)

In [605]:
finale_clus['KmeansClust'].value_counts()

0    1126
5    1057
3     924
4     701
7     693
1     668
8     603
2     277
6     183
Name: KmeansClust, dtype: int64

In [643]:
label_list[6][label_list[6]!=0]

array([-1, -1, -1, ..., -1, -1, -1])

In [647]:
np.max(label_list[7])

0

In [648]:
np.min(label_list[7])

0

In [649]:
import dill                            #pip install dill --user
filename = 'updatedClusSave.pkl'
dill.dump_session(filename)

In [1]:
import dill

dill.load_session('./updatedClusSave.pkl')

In [12]:
set(label_list[65])

{0, 1, 2, 3, 4, 5, 6, 7, 8}

In [622]:
train_set.to_csv('train_set_NOV2023.csv', index = False)
test_set.to_csv('test_set_NOV2023.csv', index = False)

In [674]:
finale_clus[finale_clus['Race Name'].str.contains('Roubaix')].head(50)

,Race Name,Constrained,KmeansClust
960,2018 Paris - Roubaix Espoirs,6,3
961,2018 Paris - Roubaix Juniors,6,3
962,2018 Paris - Roubaix,6,3
2132,2019 Paris - Roubaix Espoirs,6,3
2133,2019 Paris - Roubaix Juniors,6,3
2145,2019 Paris-Roubaix,6,3
3652,2021 Paris - Roubaix Juniors,6,3
4249,2022 Paris - Roubaix Juniors,6,3
4256,2022 Paris-Roubaix Femmes,6,3
4562,2017 Paris - Roubaix,6,3


In [676]:
finale_clus[finale_clus['KmeansClust']==8].tail(50)

,Race Name,Constrained,KmeansClust
5740,2019 Tour of Peninsula Stage 5,2,8
5745,2019 Tour of Rhodes Stage 1,2,8
5748,2019 Tour of Slovenia Stage 2,2,8
5751,2019 Tour of Slovenia Stage 5,2,8
5770,2019 Volta a Portugal Santander Stage 8,2,8
5771,2019 Volta ao Algarve em Bicicleta Stage 1,2,8
5772,2019 Volta Ciclista a Catalunya Stage 2,2,8
5775,2019 VOO-Tour de Wallonie Stage 3,2,8
5787,2019 Vuelta Ciclista a la Region de Murcia Cos...,2,8
5790,2019 Vuelta Ciclista a Venezuela Stage 7,1,8


In [621]:
finale_clus['KmeansClust'].value_counts()[0]/finale_clus.shape[0]

0.1806803594351733

In [636]:
acc_cop_kmeans = []
for i in range(7):
    metric = []
    for key, value in Counter(results_list_constrained[i]).items():
        metric.append(value/len(results_list_constrained[i]))
    acc_cop_kmeans.append(np.max(metric)) 

In [637]:
acc_cop_kmeans

[0.6830872913992297,
 0.6933568677792041,
 0.38446726572528883,
 0.25625802310654683,
 0.43469191270860075,
 0.2798459563543004,
 0.35109114249037227]

In [275]:
labeled_df = pd.DataFrame(np.matrix(label_list).transpose())
labeled_df['Race Name'] = race_names

In [276]:
labeled_df

,0,1,2,3,4,5,6,7,8,9,...,61,62,63,64,65,66,67,68,69,Race Name
0,0,78,0,0,2,0,-1,0,2,1,...,78,0,4,0,0,-1,0,5,8,2017 4 Jours de Dunkerque
1,0,11,0,0,0,0,-1,0,0,0,...,11,0,0,1,0,-1,0,1,0,2017 99a Coppa Bernocchi - 42o GP BPM
2,1,56,0,0,2,0,-1,0,2,0,...,56,0,0,0,0,-1,0,5,6,2017 Abu Dhabi Tour Stage 1
3,2,45,0,1,1,0,-1,0,1,2,...,45,0,5,2,0,-1,0,2,2,2017 Abu Dhabi Tour Stage 2
4,1,4,0,0,0,0,-1,0,2,1,...,4,0,3,8,0,-1,0,5,5,2017 Abu Dhabi Tour Stage 3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6227,1,10,0,0,2,0,-1,0,2,1,...,10,0,4,6,0,-1,0,3,8,2022 ZLM Tour Stage 1
6228,1,5,0,0,2,0,-1,0,2,0,...,5,0,4,0,0,-1,0,5,0,2022 ZLM Tour Stage 2
6229,0,78,0,0,2,0,-1,0,2,1,...,78,0,0,0,0,-1,0,5,0,2022 ZLM Tour Stage 3
6230,1,10,0,0,2,0,-1,0,2,1,...,10,0,4,6,0,-1,0,3,8,2022 ZLM Tour Stage 4


In [277]:
test_set = pd.read_csv('feitelijke_test_set2.csv')

In [286]:
test_set

,Profile1,Profile2,Label,name2_x,index_x,name2_y,index_y,Race Name_x,Distance_x,Street_x,...,Compacted Gravel_y,Off-grid (unknown)_y,Singletrack_y,Access Road_y,Alpine_y,Net Gain_y,Lowest Elevation_y,Highest Elevation_y,Vertical Gain_y,Downhill_y
102,6658,7457,MustLink,2020 Il Lombardia,6658,2021 Il Lombardia,7457,2020 Il Lombardia,231.000,0.000,...,0.00,0.000,0.000,0.0,0.0,-40.0,200.0,1261.0,4940.0,4980.0
59,7032,6212,CannotLink,2020 Vuelta a Espana Stage 3,7032,2019 Tre Valli Varesine,6212,2020 Vuelta a Espana Stage 3,165.979,0.789,...,0.00,1.420,0.000,0.0,0.0,160.0,209.0,579.0,3140.0,2980.0
94,3817,7371,CannotLink,2018 Il Lombardia,3817,2021 Giro d'Italia Stage 21,7371,2018 Il Lombardia,240.000,0.000,...,0.00,0.000,0.000,0.0,0.0,-30.0,120.0,180.0,60.0,90.0
15,7743,6838,MustLink,2021 Tirreno-Adriatico Stage 4,7743,2020 Tirreno Adriatico Stage 5,6838,2021 Tirreno-Adriatico Stage 4,148.098,0.338,...,0.00,0.000,0.000,0.0,0.0,640.0,201.0,1337.0,4370.0,3730.0
55,8747,7032,MustLink,2022 UAE Tour Stage 7,8747,2020 Vuelta a Espana Stage 3,7032,2022 UAE Tour Stage 7,148.066,0.766,...,0.00,5.190,0.000,0.0,0.0,1380.0,310.0,1718.0,2630.0,1250.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37,6240,6970,MustLink,2019 UAE Tour Stage 3,6240,2020 UAE Tour Stage 3,6970,2019 UAE Tour Stage 3,179.336,0.489,...,0.00,5.780,0.000,0.0,0.0,950.0,50.0,1032.0,1310.0,360.0
128,2808,3757,MustLink,2017 Grand Prix Cycliste de Montreal,2808,2018 Grand Prix Cycliste de Montreal,3757,2017 Grand Prix Cycliste de Montreal,212.000,0.000,...,0.00,0.000,0.000,0.0,0.0,0.0,61.0,207.0,3780.0,3780.0
58,7032,7037,MustLink,2020 Vuelta a Espana Stage 3,7032,2020 Vuelta a Espana Stage 8,7037,2020 Vuelta a Espana Stage 3,165.979,0.789,...,0.00,0.853,0.425,0.0,0.0,1020.0,412.0,1489.0,3020.0,2000.0
99,5363,2841,CannotLink,2019 Il Lombardia,5363,2017 Kampioenschap van Vlaanderen,2841,2019 Il Lombardia,243.000,0.000,...,0.00,0.000,0.000,0.0,0.0,0.0,23.0,44.0,790.0,790.0


In [ ]:
must_link = train_set[train_set['Label']=='MustLink']
cannot_link = train_set[train_set['Label']=='CannotLink']

nieuwe_tussen = df['Race Name']
nieuwe_tussen = pd.DataFrame(nieuwe_tussen)
nieuwe_tussen['index2'] = range(nieuwe_tussen.shape[0])

In [282]:
final_test = annotation[['Profile1', 'Profile2', 'Label', 'name2_x', 'name2_y']].merge(test_set[['Profile1', 'Profile2', 'Label', 'name2_x', 'name2_y']])

,Profile1,Profile2,Label,name2_x,name2_y
0,2720,2949,MustLink,2017 Dwars Door Vlaanderen,2017 Ronde Van Vlaanderen
1,2949,4088,MustLink,2017 Ronde Van Vlaanderen,2018 Ronde van Vlaanderen Beloften
2,2949,5768,CannotLink,2017 Ronde Van Vlaanderen,2019 Tirreno-Adriatico Stage 4
3,4088,4089,MustLink,2018 Ronde van Vlaanderen Beloften,2018 Ronde van Vlaanderen Juniores
4,4088,5769,CannotLink,2018 Ronde van Vlaanderen Beloften,2019 Tirreno-Adriatico Stage 5
...,...,...,...,...,...
95,6547,7292,MustLink,2020 Dwars door het Hageland,2021 Dwars door het Hageland
96,7292,8201,MustLink,2021 Dwars door het Hageland,2022 Dwars door het Hageland
97,8201,3396,MustLink,2022 Dwars door het Hageland,2018 Antwerp Port Epic
98,3396,4884,MustLink,2018 Antwerp Port Epic,2019 Antwerp Port Epic - Sels Trophy


In [280]:
test_set[['Profile1', 'Profile2', 'Label', 'name2_x', 'name2_y']].merge(annotation)

,Profile1,Profile2,Label,name2_x,name2_y,index_x,index_y,Race Name_x,Distance_x,Street_x,...,Compacted Gravel_y,Off-grid (unknown)_y,Singletrack_y,Access Road_y,Alpine_y,Net Gain_y,Lowest Elevation_y,Highest Elevation_y,Vertical Gain_y,Downhill_y
0,3264,4703,MustLink,2017 Volta Limburg Classic,2018 Volta Limburg Classic,3264,4703,2017 Volta Limburg Classic,199.000,0.00,...,0.0,0.000,0.000,0.0,0.0,0.0,48.0,285.0,2470.0,2470.0
1,5840,5834,MustLink,2019 Tour de France Stage 18,2019 Tour de France Stage 12,5840,5834,2019 Tour de France Stage 18,208.000,0.00,...,0.0,0.588,0.000,0.0,0.0,390.0,155.0,1573.0,3170.0,2780.0
2,3498,5016,MustLink,2018 Clasica Ciclista San Sebastian,2019 Clasica Ciclista San Sebastian,3498,5016,2018 Clasica Ciclista San Sebastian,228.000,0.00,...,0.0,0.000,0.000,0.0,0.0,-30.0,1.0,548.0,3980.0,4010.0
3,4628,6213,MustLink,2018 Tro-Bro Leon,2019 Tro Bro Leon,4628,6213,2018 Tro-Bro Leon,203.000,0.00,...,0.0,0.000,0.000,0.0,0.0,-20.0,0.0,134.0,1490.0,1510.0
4,3758,5285,MustLink,2018 Grand Prix Cycliste de Quebec,2019 Grand Prix Cycliste de Quebec,3758,5285,2018 Grand Prix Cycliste de Quebec,207.000,0.00,...,0.0,0.000,0.000,0.0,0.0,-10.0,5.0,101.0,2660.0,2670.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2828,6588,CannotLink,2017 Il Lombardia,2020 Giro d'Italia Stage 14,2828,6588,2017 Il Lombardia,247.000,0.00,...,0.0,0.177,0.000,0.0,0.0,180.0,61.0,352.0,490.0,310.0
96,5037,7232,MustLink,2019 Coppa Agostoni - Giro delle Brianze,2021 Coppa Agostoni - Giro delle Brianze,5037,7232,2019 Coppa Agostoni - Giro delle Brianze,192.888,4.70,...,0.0,0.000,0.000,0.0,0.0,-10.0,184.0,539.0,2890.0,2900.0
97,5285,2655,MustLink,2019 Grand Prix Cycliste de Quebec,2017 Bretagne Classic - Ouest France,5285,2655,2019 Grand Prix Cycliste de Quebec,207.000,0.00,...,0.0,0.946,0.000,0.0,0.0,0.0,41.0,291.0,2920.0,2920.0
98,6490,7190,MustLink,2020 Bretagne Classic - Ouest France,2021 Bretagne Classic - Ouest-France,6490,7190,2020 Bretagne Classic - Ouest France,248.580,2.86,...,0.0,0.000,0.000,0.0,0.0,0.0,6.0,244.0,3150.0,3150.0


In [266]:
label_list

[array([0, 0, 1, ..., 0, 1, 1]),
 array([78, 11, 56, ..., 78, 10, 34]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([2, 0, 2, ..., 2, 2, 2]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([-1, -1, -1, ..., -1, -1, -1]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([2, 0, 2, ..., 2, 2, 2]),
 array([1, 0, 0, ..., 1, 1, 1], dtype=int64),
 array([1, 2, 1, ..., 2, 1, 1]),
 array([78, 11, 56, ..., 78, 10, 34]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([3, 1, 3, ..., 3, 3, 3]),
 array([0, 2, 0, ..., 0, 0, 0]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([-1, -1, -1, ..., -1, -1, -1]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([2, 1, 2, ..., 2, 2, 2]),
 array([2, 0, 0, ..., 2, 2, 2], dtype=int64),
 array([1, 3, 1, ..., 1, 1, 1]),
 array([78, 11, 56, ..., 78, 10, 34]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([0, 4, 0, ..., 0, 0, 0]),
 array([1, 2, 1, ..., 1, 1, 1]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([-1, -1, -1, ..., -1, -1, -1]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([0, 1, 

In [265]:
df[race_names=='2017 Tour des Fjords Stage 3']['Distance']

507    180.0
Name: Distance, dtype: float64

In [ ]:
df2017 Tour des Fjords Stage 3 

In [248]:
structured_missing

,Number Category 4,Number Category 3,Number Category 2,Number Category 1,Number Hors Category,Location Last Category 4,Location Last Category 3,Location Last Category 2,Location Last Category 1,Location Last Hors Category,...,Off-grid (unknown),Singletrack,Access Road,Alpine,Net Gain,Lowest Elevation,Highest Elevation,Vertical Gain,Downhill,nas
0,1,0,0,0,0,0.500632,0.000000,0.000000,0.000000,0.000000,...,0.831,NaN,NaN,NaN,40.0,1.0,92.0,690.0,650.0,4
1,8,8,0,0,0,0.791565,0.791565,0.000000,0.000000,0.000000,...,0.366,NaN,NaN,NaN,-10.0,180.0,395.0,1880.0,1890.0,6
2,1,0,0,0,0,0.666667,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,-10.0,97.0,201.0,490.0,500.0,11
4,1,1,1,1,1,0.993711,0.993711,0.993711,0.993711,0.993711,...,1.080,NaN,0.309,NaN,770.0,154.0,1026.0,1130.0,360.0,6
5,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,...,115.000,NaN,NaN,NaN,0.0,0.0,0.0,300.0,300.0,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6219,3,2,0,0,0,0.784305,0.784305,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,0.0,9.0,249.0,1190.0,1190.0,10
6220,5,2,0,0,0,0.870857,0.649810,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,-20.0,69.0,358.0,1720.0,1740.0,10
6221,6,3,1,0,0,0.999558,0.999558,0.999558,0.000000,0.000000,...,NaN,NaN,NaN,NaN,470.0,5.0,477.0,1710.0,1240.0,10
6223,3,3,1,0,0,0.905322,0.905322,0.748448,0.000000,0.000000,...,0.316,NaN,NaN,NaN,0.0,3.0,329.0,1520.0,1520.0,8


In [245]:
all_missing

,Number Category 4,Number Category 3,Number Category 2,Number Category 1,Number Hors Category,Location Last Category 4,Location Last Category 3,Location Last Category 2,Location Last Category 1,Location Last Hors Category,...,Off-grid (unknown),Singletrack,Access Road,Alpine,Net Gain,Lowest Elevation,Highest Elevation,Vertical Gain,Downhill,nas
3,0,0,0,0,0,0.000000,0.000000,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,24.0,240.0,240.0,15
16,14,2,0,0,0,0.988737,0.799008,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,90.0,41.0,326.0,3240.0,3150.0,15
19,6,3,0,0,0,0.999810,0.999810,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,240.0,1.0,274.0,1680.0,1440.0,15
20,8,1,0,0,0,0.977924,0.481013,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,10.0,1.0,174.0,1520.0,1510.0,15
21,3,0,0,0,0,0.334859,0.000000,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,3.0,110.0,1140.0,1140.0,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6226,0,0,0,0,0,0.000000,0.000000,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,-10.0,18.0,46.0,260.0,270.0,15
6228,0,0,0,0,0,0.000000,0.000000,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,-2.0,10.0,210.0,210.0,15
6229,4,1,0,0,0,0.601414,0.535130,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,22.0,289.0,1350.0,1350.0,15
6230,0,0,0,0,0,0.000000,0.000000,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,20.0,6.0,27.0,360.0,340.0,15


In [241]:
cluster_df[cluster_df['nas']<15&cluster_df.isna()] = 0

C:\Users\banjanss\AppData\Local\Temp\ipykernel_11992\1635542697.py:1: FutureWarning: Automatic reindexing on DataFrame vs Series comparisons is deprecated and will raise ValueError in a future version. Do `left, right = left.align(right, axis=1, copy=False)` before e.g. `left == right`
  cluster_df[cluster_df['nas']<15&cluster_df.isna()] = 0


TypeError: Cannot do inplace boolean setting on mixed-types with a non np.nan value

In [234]:
cluster_df.isnull().sum(axis=0)

Number Category 4                 0
Number Category 3                 0
Number Category 2                 0
Number Category 1                 0
Number Hors Category              0
Location Last Category 4          0
Location Last Category 3          0
Location Last Category 2          0
Location Last Category 1          0
Location Last Hors Category       0
Race Name                         0
Unnamed: 0                        0
Distance                          3
Street                         2815
Road                           1964
Paved                          2439
Asphalt                        1792
Path                           5496
Cycleway                       4189
Unpaved                        5415
State Road                     2728
Cobblestones                   5504
Unknown                        3752
Compacted Gravel               6117
Off-grid (unknown)             4009
Singletrack                    6057
Access Road                    6138
Alpine                      

In [226]:
cluster_df[cluster_df['Downhill'].isna()]['Race Name']

33                         2017 BeNe Ladies Tour Prologue
295                    2017 Osterreich Rundfahrt Prologue
403          2017 The Larry H.Miller Tour of Utah Stage 3
562                            2017 Tour of Japan Stage 5
872                 2018 Carpathian Couriers Race Stage 3
967                        2018 Dookola Mazowsza Prologue
1013                    2018 Five Rings of Moscow Stage 1
1035                          2018 Gemec Nagydij Prologue
1045              2018 Giro Ciclistico d' Italia Prologue
1098                  2018 Giro della Lunigiana Stage 1-2
1103    2018 Giro Toscana Int. Femminile - Memorial Mi...
1243    2018 Istarsko Proljece - Istrian Spring Trophy...
1585                             2018 Tour Alsace Stage 1
1597    2018 Tour Cycliste International de la Guadelo...
1600    2018 Tour Cycliste International de la Guadelo...
1680                         2018 Tour de Kumano Prologue
1743                        2018 Tour de Romandie Stage 3
1860          

In [215]:
new_features

,Number Category 4,Number Category 3,Number Category 2,Number Category 1,Number Hors Category,Location Last Category 4,Location Last Category 3,Location Last Category 2,Location Last Category 1,Location Last Hors Category,Race Name
0,1,0,0,0,0,0.500632,0.000000,0.000000,0.000000,0.000000,2017 4 Jours de Dunkerque
1,8,8,0,0,0,0.791565,0.791565,0.000000,0.000000,0.000000,2017 99a Coppa Bernocchi - 42o GP BPM
2,1,0,0,0,0,0.666667,0.000000,0.000000,0.000000,0.000000,2017 Abu Dhabi Tour Stage 1
3,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,2017 Abu Dhabi Tour Stage 2
4,1,1,1,1,1,0.993711,0.993711,0.993711,0.993711,0.993711,2017 Abu Dhabi Tour Stage 3
...,...,...,...,...,...,...,...,...,...,...,...
6718,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,2022 ZLM Tour Stage 1
6719,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,2022 ZLM Tour Stage 2
6720,4,1,0,0,0,0.601414,0.535130,0.000000,0.000000,0.000000,2022 ZLM Tour Stage 3
6721,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,2022 ZLM Tour Stage 4


In [68]:
df[df['Number Hors Category']>2]

,Number Category 4,Number Category 3,Number Category 2,Number Category 1,Number Hors Category,Location Last Category 4,Location Last Category 3,Location Last Category 2,Location Last Category 1,Location Last Hors Category,Race Name
173,3,3,3,3,3,0.879271,0.879271,0.879271,0.879271,0.879271,2017 Giro d'Italia Stage 16.gpx
339,5,4,4,3,3,0.930647,0.902443,0.902443,0.902443,0.902443,2017 PRO Oetztaler 5500.gpx
784,5,5,5,5,5,0.783872,0.783872,0.783872,0.783872,0.783872,2018 Africa Cup - Criterium.gpx
1856,3,3,3,3,3,0.578477,0.578477,0.578477,0.578477,0.578477,2018 Tour of Antalya Stage 2.gpx
2606,5,4,3,3,3,0.949655,0.949655,0.699178,0.699178,0.699178,2019 Giro d'Italia Stage 20.gpx
3238,4,4,3,3,3,0.922432,0.922432,0.922432,0.922432,0.922432,2019 Tour de France Stage 18.gpx
3985,4,3,3,3,3,0.749808,0.749808,0.749808,0.749808,0.749808,2020 Giro d'Italia Stage 18.gpx
6033,7,4,4,3,3,0.799034,0.627713,0.627713,0.627713,0.627713,2022 Mont Ventoux Denivele Challenge.gpx
7161,5,4,4,4,4,0.840144,0.840144,0.840144,0.840144,0.840144,2023 International Tour of Rhodes.gpx


In [36]:
number_cat4 = len(find_peaks(track_elevation_series, height = 80)[1]['peak_heights'])
number_cat3 = len(find_peaks(track_elevation_series, height = 160)[1]['peak_heights'])
number_cat2 = len(find_peaks(track_elevation_series, height = 320)[1]['peak_heights'])
number_cat1 = len(find_peaks(track_elevation_series, height = 640)[1]['peak_heights'])
number_hc = len(find_peaks(track_elevation_series, height = 800)[1]['peak_heights'])

try:
    loc_last_cat4 = np.max(find_peaks(track_elevation_series, height = 80)[0])/len(track_elevation_series)
except:
    loc_last_cat4 = 0
try:
    loc_last_cat3 = np.max(find_peaks(track_elevation_series, height = 160)[0])/len(track_elevation_series)
except:
    loc_last_cat3 = 0
try:
    loc_last_cat2 = np.max(find_peaks(track_elevation_series, height = 320)[0])/len(track_elevation_series)
except:
    loc_last_cat2 = 0
try:
    loc_last_cat1 = np.max(find_peaks(track_elevation_series, height = 640)[0])/len(track_elevation_series)
except:
    loc_last_cat1 = 0    
try:
    loc_last_hc = np.max(find_peaks(track_elevation_series, height = 800)[0])/len(track_elevation_series)
except:
    loc_last_hc = 0        

In [41]:
loc_last_hc

0

In [31]:
len(find_peaks(track_elevation_series, height = 160)[0])

0

In [24]:
len(track_elevation_series)

8708

In [27]:
loc_last_cat4 = np.max(find_peaks(track_elevation_series, height = 80)[0])/len(track_elevation_series)
loc_last_cat3 = np.max(find_peaks(track_elevation_series, height = 160)[0])/len(track_elevation_series)
loc_last_cat2 = np.max(find_peaks(track_elevation_series, height = 320)[0])/len(track_elevation_series)
loc_last_cat1 = np.max(find_peaks(track_elevation_series, height = 640)[0])/len(track_elevation_series)
loc_last_hc = np.max(find_peaks(track_elevation_series, height = 800)[0])/len(track_elevation_series)

0.8969912723932016

In [22]:
number_cat4

3